# Thornton Melon's Economics Lesson
## As a Linear Programming (LP) Optimization Problem

**From *Back to School* (1986)**

**This Jupyter notebook created by Troy Altus, 3/3/2026**  
**for educational/amusement purposes only**


Iconic scene: Rodney Dangerfield (Thornton Melon) crashes an economics class and absolutely destroys the professor with real-world wisdom.

**Professor (all prim and proper):** "We're going to build a factory to produce widgets..."

**Thornton (interrupting):** "What's a widget?"

**Professor:** "It's a fictional product... it doesn't matter."

**Thornton:** "It doesn't matter? It matters to me!"

Then the professor starts with his clean textbook costs... and Thornton unleashes:

- "You gotta grease the local politicians for the zoning!"
- "Kickbacks to the carpenters!"
- "Chats with the teamsters!"
- "Payoffs to the building inspectors!"
- "And the mob handles the waste disposal — I assure you it's not the Boy Scouts!"

And the killer line:

> "The Japs will kill us on the labor costs!"

We turn this rant into a PuLP linear program:
- Minimize total cost while producing ≥ 1,000 tape recorders (or widgets... whatever!)
- Include Thornton's real-world overhead costs (because the professor forgot them)
- Add Japanese competition constraint → problem becomes **infeasible** (no respect!)

Costs are scaled for readability:
- Factory size (x) in units of 10,000 sq ft
- All costs in $1,000 units
- Variable labor/production: $20/unit → 0.02 in the model

Let's see if the professor's theory survives Thornton's reality...

<img src="https://i.ytimg.com/vi/lKBbFHMEvDc/maxresdefault.jpg" 
     alt="Rodney Dangerfield as Thornton Melon passionately explaining real-world costs in the classroom scene from *Back to School* (1986)" 
     style="width:60%; max-width:600px; display:block; margin:auto;">

## Problem Statement – Linear Programming Formulation

We formulate a classic **cost-minimization linear program** (the professor would approve... until Thornton shows up):

**Decision variables** (continuous, non-negative):
- x ≥ 0 : factory size (in units of 10,000 square feet)
- p ≥ 0 : number of tape recorders (or widgets... it doesn't matter!)

**Objective function** (minimize total cost in thousands of dollars):

min 119x + 0.02p + 10

**Components of the objective:**
- **119x**   = scaled fixed setup costs + real-world friction costs  
  (greasing politicians, kickbacks, teamsters, inspectors, mob waste disposal)  
  *119 is deliberately large and round-ish — chosen to make the professor’s clean model look ridiculous when Thornton’s gritty reality hits the numbers. $119,000 of “extras” before you even turn on the machines? No respect!*
- **0.02p**  = variable production/labor cost ($20 per unit, scaled)
- **10**     = fixed architect's fee (in $1,000 units)

**Constraints:**
1. p ≥ 1000                  (must meet minimum demand — "We need 1,000 widgets... or tape recorders... whatever!")  
2. p ≤ 1000x                 (production capacity: 1,000 units per 10,000 sq ft of factory)  
3. x ≥ 0, p ≥ 0              (non-negativity — even Thornton can't produce negative widgets)

This starts as the professor’s idealized textbook model... then Thornton crashes in and ruins everything.

## PuLP Implementation – Base Model (U.S. Costs Only)

In [41]:
import pulp

# Base model – American factory with real-world costs
prob = pulp.LpProblem("Thornton_Melon_Tape_Recorders", pulp.LpMinimize)

x = pulp.LpVariable("factory_size", lowBound=0)
p = pulp.LpVariable("production", lowBound=0)

prob += 119 * x + 0.02 * p + 10, "Total_Cost"
prob += p >= 1000, "Demand"
prob += p <= 1000 * x, "Capacity"

prob.solve(pulp.PULP_CBC_CMD(msg=0))

print("Status:           ", pulp.LpStatus[prob.status])
print(f"Total Cost ($1k): {pulp.value(prob.objective):.1f}")
print(f"Factory size:     {pulp.value(x):.1f} (×10,000 sq ft)")
print(f"Produced:         {pulp.value(p):.0f} units")
if pulp.value(prob.objective) is not None:
    print(f"Avg cost / unit:  ${pulp.value(prob.objective)*1000 / pulp.value(p):.2f}")

Status:            Optimal
Total Cost ($1k): 149.0
Factory size:     1.0 (×10,000 sq ft)
Produced:         1000 units
Avg cost / unit:  $149.00


## Adding the Japanese Competition Constraint

**Thornton (leaning in, wild eyes):** "The Japs will kill us on the labor costs!"

Added constraint: average cost per unit ≤ $15  
(approximate 1986 Japanese benchmark — because they actually built things efficiently)

119x + 0.02p + 10 ≤ 0.015p  
→ 119x + 10 ≤ -0.005p

This rearranged inequality is **mathematically impossible** for any p ≥ 1000.

**Professor:** "But in theory..."  
**Thornton:** "Theory? I got no respect for theory when the numbers don't work!"

In [42]:
prob_jap = pulp.LpProblem("With_Japanese_Competition", pulp.LpMinimize)

x_j = pulp.LpVariable("factory_size_j", lowBound=0)
p_j = pulp.LpVariable("production_j", lowBound=0)

prob_jap += 119 * x_j + 0.02 * p_j + 10
prob_jap += p_j >= 1000
prob_jap += p_j <= 1000 * x_j
prob_jap += 119 * x_j + 10 <= -0.005 * p_j, "Japanese_benchmark"

status = prob_jap.solve(pulp.PULP_CBC_CMD(msg=0))

print("Status:           ", pulp.LpStatus[status])
if status != pulp.LpStatusOptimal:
    print("\nINFEASIBLE – The Japs will kill us on the labor costs!")
else:
    print(f"Total Cost ($1k): {pulp.value(prob_jap.objective):.1f}")
    print(f"Factory size:     {pulp.value(x_j):.1f}")
    print(f"Produced:         {pulp.value(p_j):.0f}")

Status:            Infeasible

INFEASIBLE – The Japs will kill us on the labor costs!


## Conclusion

The math proves Thornton Melon right:

- Realistic U.S. costs → feasible, but expensive (~$149/unit)  
- Realistic 1986 Japanese competition → **infeasible**

**Thornton (grinning):** "See? No respect for that professor's numbers!"

Best alternative? Lease the space and invest the down payment — classic Thornton wisdom.

<small>*Footnote: This notebook was created with help from Grok (by xAI) for coding, formatting, and adding extra no-respect energy.*</small>